# 05 — Generate Dataset

**PLTMH–ELC–QLSTM**

Notebook ini menangani tahap **pembentukan skenario, rekonstruksi
target gain PI yang telah dibekukan, dan pembentukan trajectory
temporal mentah** sebelum tahap preprocessing.

## Batas ilmiah notebook

Notebook 05 **berakhir** ketika trajectory temporal mentah untuk
setiap skenario telah tersedia bersama metadata target
\(K_p,K_i\).

Notebook ini **tidak** melakukan:

- reduksi/seleksi feature untuk model akhir;
- pembentukan causal sliding window;
- pembagian train/validation/test;
- fitting scaler;
- normalisasi;
- pelatihan LSTM atau QLSTM.

Semua proses tersebut dimulai pada
`06_preprocessing.ipynb`.

---

### Catatan rekonstruksi

Notebook asli yang digunakan selama pengembangan menjadi notebook
Colab yang sangat panjang dan riwayat Cell 1–138 tidak lagi tersedia.
Karena itu, migrasi ini **tidak menebak urutan Cell lama**.

Target gain keluarga direkonstruksi dari bukti ilmiah yang telah
dibekukan pada workflow final. Notebook kemudian memverifikasi target
tersebut terhadap artefak repository.

**Tidak dilakukan post-result tuning.**


## 1. Kebijakan Reproduksibilitas

Dua mode dibedakan secara eksplisit:

**Mode verifikasi (default)**

- memeriksa source code;
- membangun katalog skenario;
- memverifikasi registry label;
- tidak menjalankan 120 simulasi.

**Mode reproduksi eksplisit**

Pengguna harus sengaja mengubah:

```python
REPRODUCE_RAW_TRAJECTORIES = True
```

sebelum trajectory mentah dibangkitkan kembali.

Kebijakan ini mencegah migrasi notebook secara tidak sengaja
menjalankan ulang eksperimen yang sudah dibekukan.

### Catatan mengenai gain search

`src/data/logger.py` masih memuat default gain grid historis sempit.
Default tersebut **bukan** sumber target akhir yang dipakai pada
dataset final dan karena itu tidak digunakan untuk merekonstruksi
target penelitian.

Target final yang digunakan notebook ini adalah target keluarga
yang telah dibekukan pada workflow penelitian final.


In [ ]:
# ============================================================
# 05.1 — PROJECT SETUP + SOURCE INTEGRITY
# ============================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd


EXPECTED_BUILD_CHECKPOINT = "56f8abfd5a67c6670bd4104181cd91021692515e"

EXPECTED_SOURCE_SHA256 = {
    "src/control/pi_controller.py": "7722548bb98edf994252ace0eb969b425647a01ce05d09038ec29fd5ed82cfb1",
    "src/data/logger.py": "35fae416b60bccdb2cee5597d8f811ba8ccb95aaa9ae84145cd7787b26e391c0",
    "src/plant/dump_load.py": "4c49962c3c79e6efda9424e7e83aea1b241344968741c607e4de77208128d5b0",
    "src/plant/elc.py": "98506fddd6e760a2d5075fd7f521876784c41f37959406b72e23df76095784d5",
    "src/plant/synchronous_generator.py": "1ec3f5e26bab726afddfc38e7fab2931b74068bc02cb19be280cb6a03d76157b",
    "src/simulation/scenario.py": "973bc400a8e0b542c1837f940c5989e0203805bb2ff74b706d058675e6a23cb8"
}

EXPECTED_LABEL_EVIDENCE_SHA256 = (
    "63cea54cdcb1a9922640ab2cbd387642ff79b00ae731540bf119680858ac595f"
)

EXPECTED_FINAL_METADATA_SHA256 = (
    "0e11a631937bbe349fe90bf326ba97fd5e14d209859ad10ca159b6fdf74efcf5"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "src").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print("Project root:", PROJECT_ROOT)
print("Build checkpoint:", EXPECTED_BUILD_CHECKPOINT)


source_integrity = {}


for relative_path, expected_sha in (
    EXPECTED_SOURCE_SHA256.items()
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )

    current_sha = sha256_file(
        path
    )

    source_integrity[
        relative_path
    ] = (
        current_sha
        ==
        expected_sha
    )

    print(
        f"{relative_path:48s}: "
        f"{source_integrity[relative_path]}"
    )


if not all(
    source_integrity.values()
):

    raise RuntimeError(
        "Validated source contract changed."
    )


LABEL_EVIDENCE_PATH = (
    PROJECT_ROOT
    /
    "data/splits/cell137_family_split_assignment.csv"
)


FINAL_METADATA_PATH = (
    PROJECT_ROOT
    /
    "data/final/qlstm_lstm_dataset_metadata.json"
)


if (
    sha256_file(
        LABEL_EVIDENCE_PATH
    )
    !=
    EXPECTED_LABEL_EVIDENCE_SHA256
):

    raise RuntimeError(
        "Frozen family-label evidence changed."
    )


if (
    sha256_file(
        FINAL_METADATA_PATH
    )
    !=
    EXPECTED_FINAL_METADATA_SHA256
):

    raise RuntimeError(
        "Frozen final-dataset metadata changed."
    )


SOURCE_CONTRACT_VALID = True


print(
    "\nSOURCE_CONTRACT_VALID:",
    SOURCE_CONTRACT_VALID
)


In [ ]:
# ====================================================
# 05.2 — IMPORT VALIDATED PLTMH–ELC PHYSICS
# ====================================================

from src.simulation.scenario import (
    build_default_scenario_catalog,
)

from src.plant.synchronous_generator import (
    GeneratorParams,
)

from src.plant.elc import (
    ELCParams,
)

from src.data.logger import (
    build_controller,
)

from src.control.pi_controller import (
    simulate_pi_elc_closed_loop_rk4,
)


generator_params = GeneratorParams()

elc_params = ELCParams()


print(
    "Generator rated power [kW]:",
    generator_params.rated_power_kw
)

print(
    "Nominal frequency [Hz]:",
    generator_params.nominal_frequency_hz
)

print(
    "Inertia constant H [s]:",
    generator_params.inertia_constant_s
)

print(
    "ELC actuator tau [s]:",
    elc_params.actuator_time_constant_s
)

print(
    "Dump-load rating [kW]:",
    elc_params.dump_load.rated_power_kw
)


In [ ]:
# ====================================================
# 05.3 — BUILD THE AUTHORITATIVE 120-SCENARIO CATALOG
# ====================================================

scenarios = (
    build_default_scenario_catalog()
)


if len(scenarios) != 120:

    raise RuntimeError(
        "Expected exactly 120 DatasetScenario objects."
    )


def family_key(
    initial_dump_power_kw,
    load_step_kw,
):

    dump_code = int(
        round(
            initial_dump_power_kw
        )
    )

    load_code = int(
        round(
            load_step_kw
        )
    )

    load_text = (
        f"+{load_code}"
        if load_code > 0
        else str(load_code)
    )

    return (
        f"D{dump_code}_L{load_text}"
    )


scenario_rows = []


for scenario in scenarios:

    required_final_dump_kw = (
        scenario.mechanical_power_kw
        -
        scenario.final_consumer_power_kw
    )

    required_final_duty = (
        required_final_dump_kw
        /
        scenario.rated_power_kw
    )

    if required_final_duty <= 1e-12:

        boundary = "LOWER"

    elif required_final_duty >= 1.0 - 1e-12:

        boundary = "UPPER"

    else:

        boundary = "INTERIOR"


    scenario_rows.append(
        {
            "scenario_id":
                scenario.scenario_id,

            "family_key":
                family_key(
                    scenario.initial_dump_power_kw,
                    scenario.load_step_kw,
                ),

            "mechanical_power_pu":
                scenario.mechanical_power_pu,

            "mechanical_power_kw":
                scenario.mechanical_power_kw,

            "initial_dump_power_kw":
                scenario.initial_dump_power_kw,

            "initial_consumer_power_kw":
                scenario.initial_consumer_power_kw,

            "load_step_kw":
                scenario.load_step_kw,

            "final_consumer_power_kw":
                scenario.final_consumer_power_kw,

            "required_final_dump_power_kw":
                required_final_dump_kw,

            "required_final_duty":
                required_final_duty,

            "required_final_boundary":
                boundary,

            "disturbance_time_s":
                scenario.disturbance_time_s,
        }
    )


scenario_catalog = pd.DataFrame(
    scenario_rows
)


print(
    "Scenario count:",
    len(
        scenario_catalog
    )
)

print(
    "Dynamic family count:",
    scenario_catalog[
        "family_key"
    ].nunique()
)


print(
    "\nScenarios per family:"
)

print(
    scenario_catalog
    .groupby(
        "family_key"
    )
    .size()
    .sort_index()
)


In [ ]:
# ============================================================
# 05.4 — FROZEN FAMILY-LEVEL GAIN LABEL REGISTRY
# ============================================================

# Recovered from the finalized scientific workflow.
#
# This registry is embedded explicitly so Notebook 05 does
# NOT depend scientifically on train/validation/test split
# semantics from Notebook 06.

FROZEN_FAMILY_LABELS = [
    {
        "family_key": "D20_L-20",
        "gain_regime": "G1",
        "kp_target": 2.16,
        "ki_target": 5.72
    },
    {
        "family_key": "D30_L-20",
        "gain_regime": "G1",
        "kp_target": 2.16,
        "ki_target": 5.72
    },
    {
        "family_key": "D40_L-20",
        "gain_regime": "G1",
        "kp_target": 2.16,
        "ki_target": 5.72
    },
    {
        "family_key": "D40_L+20",
        "gain_regime": "G1",
        "kp_target": 2.16,
        "ki_target": 5.72
    },
    {
        "family_key": "D20_L-10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D20_L+10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D30_L-10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D30_L+10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D40_L-10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D40_L+10",
        "gain_regime": "G2",
        "kp_target": 2.88,
        "ki_target": 3.0368
    },
    {
        "family_key": "D30_L+20",
        "gain_regime": "G3",
        "kp_target": 3.36,
        "ki_target": 14.144
    }
]


label_registry = pd.DataFrame(
    FROZEN_FAMILY_LABELS
)


if label_registry["family_key"].duplicated().any():

    raise RuntimeError(
        "Family label registry contains duplicates."
    )


if len(label_registry) != 11:

    raise RuntimeError(
        "Expected 11 supervised dynamic families."
    )


# ------------------------------------------------------------
# Verify recovered labels against frozen repository evidence.
#
# IMPORTANT:
# The 'split' column is NOT used by Notebook 05.
# Only family_key / regime / Kp / Ki are used for migration
# verification.
# ------------------------------------------------------------

frozen_evidence = pd.read_csv(
    LABEL_EVIDENCE_PATH
)


evidence_targets = (
    frozen_evidence[
        [
            "family_key",
            "gain_regime",
            "kp_target",
            "ki_target",
        ]
    ]
    .sort_values(
        "family_key"
    )
    .reset_index(
        drop=True
    )
)


registry_targets = (
    label_registry[
        [
            "family_key",
            "gain_regime",
            "kp_target",
            "ki_target",
        ]
    ]
    .sort_values(
        "family_key"
    )
    .reset_index(
        drop=True
    )
)


same_families = bool(
    registry_targets[
        "family_key"
    ].tolist()
    ==
    evidence_targets[
        "family_key"
    ].tolist()
)


same_regimes = bool(
    registry_targets[
        "gain_regime"
    ].tolist()
    ==
    evidence_targets[
        "gain_regime"
    ].tolist()
)


same_kp = bool(
    np.allclose(
        registry_targets[
            "kp_target"
        ].to_numpy(
            dtype=float
        ),
        evidence_targets[
            "kp_target"
        ].to_numpy(
            dtype=float
        ),
        rtol=0.0,
        atol=1e-12,
    )
)


same_ki = bool(
    np.allclose(
        registry_targets[
            "ki_target"
        ].to_numpy(
            dtype=float
        ),
        evidence_targets[
            "ki_target"
        ].to_numpy(
            dtype=float
        ),
        rtol=0.0,
        atol=1e-12,
    )
)


LABEL_REGISTRY_VERIFIED = all(
    [
        same_families,
        same_regimes,
        same_kp,
        same_ki,
    ]
)


print(
    label_registry
    .sort_values(
        [
            "gain_regime",
            "family_key",
        ]
    )
    .to_string(
        index=False
    )
)


print(
    "\nRecovered family labels verified:",
    LABEL_REGISTRY_VERIFIED
)


if not LABEL_REGISTRY_VERIFIED:

    raise RuntimeError(
        "Recovered family labels do not match "
        "the frozen scientific evidence."
    )


In [ ]:
# ====================================================
# 05.5 — MAP FAMILY LABELS TO ALL 120 SCENARIOS
# ====================================================

BOUNDARY_FAMILY = "D20_L+20"


scenario_registry = (
    scenario_catalog
    .merge(
        label_registry,
        on="family_key",
        how="left",
        validate="many_to_one",
    )
)


scenario_registry[
    "supervised_eligible"
] = (
    scenario_registry[
        "family_key"
    ]
    !=
    BOUNDARY_FAMILY
)


scenario_registry[
    "dataset_role"
] = np.where(
    scenario_registry[
        "supervised_eligible"
    ],
    "SUPERVISED",
    "BOUNDARY_STRESS_UNLABELED",
)


supervised_count = int(
    scenario_registry[
        "supervised_eligible"
    ].sum()
)


boundary_count = int(
    (
        ~
        scenario_registry[
            "supervised_eligible"
        ]
    ).sum()
)


if supervised_count != 110:

    raise RuntimeError(
        "Expected 110 supervised scenarios."
    )


if boundary_count != 10:

    raise RuntimeError(
        "Expected 10 boundary scenarios."
    )


if not (
    scenario_registry.loc[
        ~
        scenario_registry[
            "supervised_eligible"
        ],
        "family_key",
    ]
    ==
    BOUNDARY_FAMILY
).all():

    raise RuntimeError(
        "Boundary family mismatch."
    )


if (
    scenario_registry.loc[
        scenario_registry[
            "supervised_eligible"
        ],
        [
            "kp_target",
            "ki_target",
            "gain_regime",
        ],
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "A supervised scenario has a missing label."
    )


print(
    "Total scenarios:",
    len(
        scenario_registry
    )
)

print(
    "Supervised scenarios:",
    supervised_count
)

print(
    "Boundary/unlabeled scenarios:",
    boundary_count
)


print(
    "\nScenario count by regime:"
)

regime_summary = (
    scenario_registry
    .assign(
        resolved_regime=lambda df:
            df[
                "gain_regime"
            ].fillna(
                "BOUNDARY_UNLABELED"
            )
    )
    .groupby(
        "resolved_regime"
    )
    .size()
)


print(
    regime_summary
)


EXPECTED_REGIME_COUNTS = {
    "G1": 40,
    "G2": 60,
    "G3": 10,
    "BOUNDARY_UNLABELED": 10,
}


if (
    regime_summary.to_dict()
    !=
    EXPECTED_REGIME_COUNTS
):

    raise RuntimeError(
        "Gain-regime scenario counts changed."
    )


SCENARIO_LABEL_CONTRACT_VALID = True


print(
    "\nSCENARIO_LABEL_CONTRACT_VALID:",
    SCENARIO_LABEL_CONTRACT_VALID
)


## 2. Kebijakan Trajectory Mentah

Target \(K_p,K_i\) merupakan **label supervisi**, tetapi target tersebut
**tidak digunakan untuk mengendalikan plant ketika sinyal masukan temporal
dibangkitkan**.

Seluruh trajectory mentah menggunakan satu **common source PI**:

\[
K_p = 2.16, \qquad K_i = 5.72
\]

dengan `output_bias` tetap mengikuti kondisi dump-load awal setiap
skenario.

Kebijakan ini penting untuk mencegah **target-conditioned input leakage**:
sinyal masukan LSTM/QLSTM tidak boleh sudah membawa jejak dinamik dari
gain target yang hendak diprediksi.

Solver reproduksi trajectory menggunakan fixed-step RK4:

\[
\Delta t = 0.0025\;\text{s}
\]

selama 10 s. Resampling 20 Hz, feature selection, causal windowing,
dan split keluarga merupakan tanggung jawab Notebook 06.


In [ ]:
# ====================================================
# 05.6 — COMMON-SOURCE RAW TRAJECTORY GENERATOR
# ====================================================

SOURCE_PI_KP = 2.16

SOURCE_PI_KI = 5.72

RAW_RK4_DT_S = 0.0025

RAW_SIMULATION_END_S = 10.0


def generate_common_source_trajectory(
    scenario,
    registry_row,
):

    # ------------------------------------------------
    # Common source PI:
    # target labels are metadata only.
    # ------------------------------------------------

    controller = build_controller(
        scenario=scenario,
        generator_params=generator_params,
        kp=SOURCE_PI_KP,
        ki=SOURCE_PI_KI,
    )


    result = (
        simulate_pi_elc_closed_loop_rk4(
            controller_params=controller,
            generator_params=generator_params,
            elc_params=elc_params,

            mechanical_power_profile=(
                scenario
                .mechanical_power_profile()
            ),

            consumer_power_profile=(
                scenario
                .consumer_power_profile()
            ),

            t_end_s=RAW_SIMULATION_END_S,
            dt_s=RAW_RK4_DT_S,

            initial_omega_pu=1.0,
            initial_delta_rad=0.0,
            initial_integral_error_hz_s=0.0,

            initial_dump_power_kw=(
                scenario
                .initial_dump_power_kw
            ),
        )
    )


    result = result.copy()


    # ------------------------------------------------
    # Exact raw feature aliases later consumed by
    # preprocessing.
    # ------------------------------------------------

    result[
        "frequency_deviation_hz"
    ] = (
        result[
            "frequency_hz"
        ]
        -
        generator_params
        .nominal_frequency_hz
    )


    result[
        "mechanical_power_kw"
    ] = (
        result[
            "pm_pu"
        ]
        *
        generator_params
        .rated_power_kw
    )


    # electrical_power_kw already exists
    # dump_power_kw already exists


    # ------------------------------------------------
    # Scenario metadata
    # ------------------------------------------------

    result[
        "scenario_id"
    ] = scenario.scenario_id


    result[
        "family_key"
    ] = registry_row[
        "family_key"
    ]


    result[
        "mechanical_power_scenario_pu"
    ] = scenario.mechanical_power_pu


    result[
        "initial_dump_power_kw"
    ] = scenario.initial_dump_power_kw


    result[
        "load_step_kw"
    ] = scenario.load_step_kw


    result[
        "disturbance_time_s"
    ] = scenario.disturbance_time_s


    result[
        "time_since_disturbance_s"
    ] = (
        result[
            "time_s"
        ]
        -
        scenario.disturbance_time_s
    )


    # ------------------------------------------------
    # Supervised label metadata.
    #
    # Boundary D20_L+20 remains NaN by design.
    # ------------------------------------------------

    result[
        "gain_regime"
    ] = registry_row[
        "gain_regime"
    ]


    result[
        "kp_target"
    ] = registry_row[
        "kp_target"
    ]


    result[
        "ki_target"
    ] = registry_row[
        "ki_target"
    ]


    result[
        "supervised_eligible"
    ] = bool(
        registry_row[
            "supervised_eligible"
        ]
    )


    result[
        "dataset_role"
    ] = registry_row[
        "dataset_role"
    ]


    return result


print(
    "Common source PI:",
    (
        SOURCE_PI_KP,
        SOURCE_PI_KI,
    )
)

print(
    "Raw RK4 dt [s]:",
    RAW_RK4_DT_S
)

print(
    "Trajectory generator defined:",
    True
)


In [ ]:
# ====================================================
# 05.7 — OPTIONAL RAW-DATA REPRODUCTION
# ====================================================
#
# SAFE DEFAULT:
#
# Migration / verification MUST NOT rerun 120
# simulations automatically.
#
# Change to True ONLY for an explicit scientific
# reproduction run.
# ====================================================

REPRODUCE_RAW_TRAJECTORIES = False


RAW_OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "raw"
)


RAW_TRAJECTORY_PATH = (
    RAW_OUTPUT_DIR
    /
    "notebook05_raw_temporal_trajectories.csv.gz"
)


SCENARIO_REGISTRY_PATH = (
    RAW_OUTPUT_DIR
    /
    "notebook05_scenario_label_registry.csv"
)


RAW_METADATA_PATH = (
    RAW_OUTPUT_DIR
    /
    "notebook05_raw_generation_metadata.json"
)


RAW_REPRODUCTION_PERFORMED = False


if not REPRODUCE_RAW_TRAJECTORIES:

    print(
        "REPRODUCE_RAW_TRAJECTORIES = False"
    )

    print(
        "No 120-scenario simulation was run."
    )

    print(
        "This is the expected migration/default mode."
    )


else:

    RAW_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    scenario_lookup = {
        scenario.scenario_id:
            scenario
        for scenario
        in scenarios
    }


    trajectory_frames = []


    for index, registry_row in (
        scenario_registry
        .sort_values(
            "scenario_id"
        )
        .iterrows()
    ):

        scenario_id = (
            registry_row[
                "scenario_id"
            ]
        )


        scenario = (
            scenario_lookup[
                scenario_id
            ]
        )


        trajectory = (
            generate_common_source_trajectory(
                scenario,
                registry_row,
            )
        )


        trajectory_frames.append(
            trajectory
        )


        if (
            len(
                trajectory_frames
            )
            %
            10
            ==
            0
        ):

            print(
                "Generated:",
                len(
                    trajectory_frames
                ),
                "/ 120"
            )


    raw_trajectories = pd.concat(
        trajectory_frames,
        ignore_index=True,
    )


    expected_rows = (
        120
        *
        4001
    )


    if len(
        raw_trajectories
    ) != expected_rows:

        raise RuntimeError(
            "Unexpected raw trajectory row count."
        )


    if not np.isfinite(
        raw_trajectories[
            [
                "time_s",
                "frequency_hz",
                "frequency_deviation_hz",
                "mechanical_power_kw",
                "electrical_power_kw",
                "dump_power_kw",
            ]
        ].to_numpy(
            dtype=float
        )
    ).all():

        raise RuntimeError(
            "Non-finite physical data detected."
        )


    raw_trajectories.to_csv(
        RAW_TRAJECTORY_PATH,
        index=False,
        compression="gzip",
    )


    scenario_registry.to_csv(
        SCENARIO_REGISTRY_PATH,
        index=False,
    )


    generation_metadata = {
        "status":
            "RAW_LABELED_TEMPORAL_TRAJECTORIES",

        "scenario_count":
            120,

        "supervised_scenario_count":
            110,

        "boundary_scenario_count":
            10,

        "raw_rows":
            int(
                len(
                    raw_trajectories
                )
            ),

        "solver":
            "FIXED_STEP_RK4",

        "solver_dt_s":
            RAW_RK4_DT_S,

        "simulation_end_s":
            RAW_SIMULATION_END_S,

        "common_source_pi":
            {
                "kp":
                    SOURCE_PI_KP,

                "ki":
                    SOURCE_PI_KI,
            },

        "target_conditioned_input_leakage":
            False,

        "feature_windowing_performed":
            False,

        "normalization_performed":
            False,

        "train_validation_test_split_performed":
            False,
    }


    RAW_METADATA_PATH.write_text(
        json.dumps(
            generation_metadata,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )


    RAW_REPRODUCTION_PERFORMED = True


    print(
        "\nRaw trajectories:",
        RAW_TRAJECTORY_PATH
    )

    print(
        "Scenario registry:",
        SCENARIO_REGISTRY_PATH
    )

    print(
        "Metadata:",
        RAW_METADATA_PATH
    )


In [ ]:
# ====================================================
# 05.8 — DOWNSTREAM FROZEN-EVIDENCE CONSISTENCY AUDIT
# ====================================================

final_metadata = json.loads(
    FINAL_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


expected_features = [
    "frequency_deviation_hz",
    "mechanical_power_kw",
    "electrical_power_kw",
    "dump_power_kw",
]


downstream_contract_ok = all(
    [
        final_metadata[
            "dataset_status"
        ]
        ==
        "FINAL_LSTM_QLSTM_READY",

        final_metadata[
            "supervised_window_count"
        ]
        ==
        17600,

        final_metadata[
            "sequence_length"
        ]
        ==
        11,

        final_metadata[
            "feature_count"
        ]
        ==
        4,

        final_metadata[
            "features"
        ]
        ==
        expected_features,

        final_metadata[
            "observation_dt_s"
        ]
        ==
        0.05,

        final_metadata[
            "boundary_stress_test_in_supervised_tensor"
        ]
        is False,
    ]
)


print(
    "Frozen downstream dataset status:",
    final_metadata[
        "dataset_status"
    ]
)

print(
    "Supervised windows:",
    final_metadata[
        "supervised_window_count"
    ]
)

print(
    "Sequence length:",
    final_metadata[
        "sequence_length"
    ]
)

print(
    "Features:",
    final_metadata[
        "features"
    ]
)

print(
    "Observation dt [s]:",
    final_metadata[
        "observation_dt_s"
    ]
)

print(
    "Boundary included in supervised tensor:",
    final_metadata[
        "boundary_stress_test_in_supervised_tensor"
    ]
)

print(
    "\nDOWNSTREAM_FROZEN_EVIDENCE_CONSISTENT:",
    downstream_contract_ok
)


if not downstream_contract_ok:

    raise RuntimeError(
        "Notebook-05 reconstruction conflicts "
        "with the frozen downstream dataset."
    )


## 3. Handoff ke `06_preprocessing.ipynb`

Kontrak keluaran Notebook 05 adalah:

**RAW_LABELED_TEMPORAL_TRAJECTORIES**

Setiap trajectory membawa:

- `scenario_id`;
- `family_key`;
- waktu;
- deviasi frekuensi;
- daya mekanik;
- daya elektrik;
- daya dump;
- metadata gangguan;
- `kp_target`;
- `ki_target`;
- `gain_regime`;
- flag `supervised_eligible`.

Keluarga `D20_L+20` tetap dipertahankan sebagai **boundary stress**
dan tidak mempunyai target supervised primer.

Tahapan berikut **bukan bagian Notebook 05**:

1. observasi/resampling 20 Hz;
2. pemilihan empat feature akhir;
3. causal window 11 titik / 0,5 s;
4. family-level split;
5. scaler train-only;
6. final tensor packaging.

Tahapan tersebut dilakukan pada `06_preprocessing.ipynb`.


In [ ]:
# ====================================================
# 05.9 — NOTEBOOK-05 READINESS SUMMARY
# ====================================================

NOTEBOOK_05_DATASET_CONTRACT_READY = all(
    [
        SOURCE_CONTRACT_VALID,
        LABEL_REGISTRY_VERIFIED,
        SCENARIO_LABEL_CONTRACT_VALID,
        downstream_contract_ok,
    ]
)


print("=" * 72)
print("05_generate_dataset.ipynb — SUMMARY")
print("=" * 72)

print(
    "Validated source contract           :",
    SOURCE_CONTRACT_VALID
)

print(
    "Scenario catalogue                  :",
    len(
        scenario_registry
    )
)

print(
    "Supervised scenarios                :",
    int(
        scenario_registry[
            "supervised_eligible"
        ].sum()
    )
)

print(
    "Boundary scenarios                  :",
    int(
        (
            ~
            scenario_registry[
                "supervised_eligible"
            ]
        ).sum()
    )
)

print(
    "Supervised families                 :",
    len(
        label_registry
    )
)

print(
    "Boundary family                     :",
    BOUNDARY_FAMILY
)

print(
    "Common source PI                    :",
    (
        SOURCE_PI_KP,
        SOURCE_PI_KI,
    )
)

print(
    "Raw RK4 dt [s]                      :",
    RAW_RK4_DT_S
)

print(
    "Target-conditioned leakage          : False"
)

print(
    "Preprocessing performed here        : False"
)

print(
    "Raw reproduction requested          :",
    REPRODUCE_RAW_TRAJECTORIES
)

print(
    "Raw reproduction performed          :",
    RAW_REPRODUCTION_PERFORMED
)

print(
    "NOTEBOOK 05 DATASET CONTRACT READY  :",
    NOTEBOOK_05_DATASET_CONTRACT_READY
)


if NOTEBOOK_05_DATASET_CONTRACT_READY:

    print(
        "\nNEXT NOTEBOOK:"
    )

    print(
        "06_preprocessing.ipynb"
    )
